# Insurance Claims — Volume Forecast (CAI Workbench)

**Goal:** Use gold-layer Iceberg tables in `holuser01_insurance_analytics` to forecast next-month claim volume by policy type.

**Prerequisites:** Medallion jobs through `07_gold_trends_and_watchlist` completed.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lag, avg as spark_avg
from pyspark.sql.window import Window
import pandas as pd

spark = (
    SparkSession.builder.appName("holuser01-insurance-forecast")
    .enableHiveSupport()
    .getOrCreate()
)

DB = "holuser01_insurance_analytics"
trends = spark.table(f"{DB}.gold_monthly_claim_trends").orderBy("claim_month", "policy_type")
trends.show(10, truncate=False)

## Feature prep: 3-month rolling average per policy type

In [ ]:
w = Window.partitionBy("policy_type").orderBy("claim_month")

features = (
    trends
    .withColumn("prev_claim_count", lag("claim_count", 1).over(w))
    .withColumn("prev2_claim_count", lag("claim_count", 2).over(w))
    .withColumn(
        "rolling_avg_3m",
        spark_avg("claim_count").over(w.rowsBetween(-2, 0)),
    )
    .dropna(subset=["rolling_avg_3m"])
)

pdf = features.select(
    "claim_month", "policy_type", "claim_count", "rolling_avg_3m"
).toPandas()
pdf.head(12)

## Simple forecast: next period = latest rolling average per policy type

Workshop-friendly baseline (no extra ML libraries required).

In [ ]:
latest = (
    pdf.sort_values("claim_month")
    .groupby("policy_type", as_index=False)
    .tail(1)
    .rename(columns={"rolling_avg_3m": "forecast_next_month_claim_count"})
)
latest[["policy_type", "claim_month", "claim_count", "forecast_next_month_claim_count"]]

## Optional: linear trend on total monthly claims (pandas)

In [ ]:
monthly_total = pdf.groupby("claim_month", as_index=False)["claim_count"].sum()
monthly_total["t"] = range(len(monthly_total))

coef = __import__("numpy").polyfit(monthly_total["t"], monthly_total["claim_count"], 1)
next_t = monthly_total["t"].max() + 1
next_forecast = coef[0] * next_t + coef[1]
print(f"Linear trend forecast — total claims next month: {next_forecast:,.0f}")
monthly_total.tail()

## Fraud segment snapshot (gold watchlist)

In [ ]:
watch = spark.table(f"{DB}.gold_high_risk_watchlist")
watch.groupBy("state").count().orderBy(col("count").desc()).show(10)
watch.orderBy(col("claim_amount").desc()).select(
    "claim_id", "state", "policy_type", "claim_amount", "fraud_risk_score"
).show(10, truncate=False)